# **ML4_Classification problems. Проблемы классификации.**

### ***1. Download data from Don’tGetKicked competition.***

***Загрузите данные соревнования "Don’tGetKicked"***

Перед началом выполнения  проекта установите все необходимые библиотеки, выполнив 
`pip install -r requirements.txt`

**IMPORT LIBRARIES. ИМПОРТ БИБЛИОТЕК.**

In [52]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import ml_utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Загрузка данных соревнования выполнена через скрипт `src/download_competition_data.py`. 
Датасет `DontGetKicked` скачан, автоматически распакован и сохранен локально в директорию `./datasets/`.

In [3]:
train_path = "../datasets/training.csv"
test_path = "../datasets/test.csv"

if os.path.exists(train_path) and os.path.exists(test_path):
    data_train = pd.read_csv(train_path)
    data_test = pd.read_csv(test_path)

    print("Данные успешно загружены локально!")
    print(f"Строк и признаков в Train: {data_train.shape}")
    print(f"Строк и признаков в Test: {data_test.shape}")
else:
    print("Ошибка: Файлы датасета не найдены по указанному пути!")
    print(
        f"Пожалуйста, проверьте наличие файлов в папке: {os.path.abspath('./datasets')}"
    )

Данные успешно загружены локально!
Строк и признаков в Train: (72983, 34)
Строк и признаков в Test: (48707, 33)


In [4]:
data_train.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


In [ ]:
# таргет
# data_train[["IsBadBuy"]]

#### ***2. Design the train/validation/test split. Use the "PurchDate" field for the split, test must be later than validation, same for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate. Use the first 1/3 of dates for the train, the last 1/3 of dates for the test, and the middle 1/3 for the validation set. Don’t use the test dataset until the end!***

*Разработай схему разбиения данных на обучающую, валидационную и тестовую выборки. Для разделения используй поле «PurchDate». Тестовая выборка должна хронологически следовать за валидационной, а валидационная - за обучающей, что выражается строгим неравенством: train.PurchDate < valid.PurchDate < test.PurchDate. Выдели первую треть (1/3) хронологического периода под обучающий набор, последнюю треть (1/3) - под тестовый, а среднюю треть - под валидационный. Помни, что тестовый датасет должен оставаться полностью изолированным вплоть до финального тестирования модели!*


In [ ]:
data_train["PurchDate"].dtype

В колонке 'PurchDate' хранятся данные в формате str


In [ ]:
data_train["PurchDate"] = pd.to_datetime(data_train["PurchDate"])
data_train["PurchDate"].dtype

dtype('<M8[us]')

In [25]:
print(
    f"В колонке 'PurchDate' хранятся данные в формате {data_train['PurchDate'].dtype}"
)

В колонке 'PurchDate' хранятся данные в формате datetime64[us]


In [ ]:
print(
    f"Минимальная дата в датасете: {data_train['PurchDate'].min().strftime('%Y-%m-%d')}"
)
print(
    f"Максимальная дата в датасете: {data_train['PurchDate'].max().strftime('%Y-%m-%d')}"
)

Минимальная дата в датасете: 2009-01-05
Максимальная дата в датасете: 2010-12-30


In [ ]:
data_train = data_train.sort_values(by="PurchDate").reset_index(drop=True)

rows_num = len(data_train)
part = rows_num // 3

In [18]:
part

24327

In [49]:
validation_date_cutoff = data_train.iloc[part]["PurchDate"]
validation_date_cutoff

Timestamp('2009-09-15 00:00:00')

In [51]:
test_date_cutoff = data_train.iloc[part * 2]["PurchDate"]
test_date_cutoff

Timestamp('2010-05-14 00:00:00')

In [54]:
train_split, valid_split, test_split = ml_utils.date_trisect(
    df=data_train,
    date_column="PurchDate",
    validation_date=validation_date_cutoff,
    test_date=test_date_cutoff,
)

print("Временной сплит:")
print(
    f"Train: {train_split.shape} | {train_split['PurchDate'].min().strftime('%Y-%m-%d')} - {train_split['PurchDate'].max().strftime('%Y-%m-%d')}"
)
print(
    f"Valid: {valid_split.shape} | {valid_split['PurchDate'].min().strftime('%Y-%m-%d')} - {valid_split['PurchDate'].max().strftime('%Y-%m-%d')}"
)
print(
    f"Test:  {test_split.shape} | {test_split['PurchDate'].min().strftime('%Y-%m-%d')} - {test_split['PurchDate'].max().strftime('%Y-%m-%d')}"
)

Временной сплит:
Train: (24232, 34) | 2009-01-05 - 2009-09-14
Valid: (24414, 34) | 2009-09-15 - 2010-05-13
Test:  (24337, 34) | 2010-05-14 - 2010-12-30


#### ***3. Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training): https://contrib.scikit-learn.org/category_encoders/count.html***

#### ***4. Train LogisticRegression, GaussianNB, KNN from sklearn on the training dataset and check the quality of your algorithms on the validation dataset. The dependent variable (IsBadBuy) is binary. Don't forget to normalize your datasets before training your models.***

***You must get at least 0.15 Gini score (the best of all three). Which algorithm performs better? And why?***

#### ***5. Implement Gini score calculation. You can use the 2*ROC AUC - 1 approach, so you need to implement the ROC AUC calculation.**
Check if your metric is approximately equal to abs(2*sklearn.metrics.roc_auc_score - 1)

#### ***6. Implement your own versions of LogisticRegression, KNN and NaiveBayes classifiers. For LogisticRegression compute gradients with respect to the loss and use stochastic gradient descent. Can you reproduce the results from step 4?***

*Guidance for this task: Your model must be represented by class with methods fit, predict (predict_proba with 0.5 threshold), predict_proba. For LR moder, compute the loss gradient with respect to parameters w and parameter b in the fit function. Use a simple SGD approach to estimate optimal values of parameters.*

#### ***7. Try to create non-linear features, for example:***

fractions: feature1/feature2
groupby features: df[‘categorical_feature’].map(df.groupby(‘categorical_feature’)[‘continious_feature’].mean())

Add new features to your pipeline, repeat step 4. Did you manage to increase your Gini score (you should!)?***

#### ***8. Determine the best features for the problem using the coefficients of the logistic model. Try to eliminate useless features by hand and by L1 regularization. Which approach is better in terms of Gini score?***

#### ***9. Select your best model (algorithm + feature set) and tweak its hyperparameters to increase the Gini score on the validation dataset. Which hyperparameters have the most impact?***

#### ***10. Check the Gini scores on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitted or not? Explain.***

#### ***11. Implement calculation of Recall, Precision, F1 score and AUC PR metrics.***
Compare your algorithms on the test dataset using AUC PR metric.

#### ***12. Which hard label metric do you prefer for the task of detecting "lemon" cars?***